# 04 — SAM/Vicon validation

Main local working notebook. Reuses `src.comparison_engine` for metrics and `src.plotting` for visualization. **Already-aligned CSVs with embedded Vicon** are legacy exploratory comparisons, not independent verification of synchronization.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "comparison_engine.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "src" / "comparison_engine.py").is_file():
    raise FileNotFoundError("Start Jupyter in markerless_sam/ or notebooks/")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "lunge_sam_out"
RESULTS = ROOT / "results"
print("Repository:", ROOT, "| Data folder available:", DATA.is_dir())


In [ ]:
from src.comparison_engine import compare_angles
from src.plotting import plot_joint_overlays, plot_joint_errors, plot_camera_comparison
import matplotlib.pyplot as plt

files = sorted(DATA.rglob("*_joint_angles.csv"))
print(f"Found {len(files)} joint-angle CSV files")
for i, path in enumerate(files):
    print(i, path.relative_to(DATA))


In [ ]:
SELECTED = 0
# Optional exploratory trial-start calibration; never infer a sync shift from joints.
STATIC_WINDOW_MS = None  # e.g. 1500, only if the beginning is known to be static

df = None
metrics = None
if files:
    if not 0 <= SELECTED < len(files):
        raise ValueError(f"SELECTED must be between 0 and {len(files) - 1}")
    csv_path = files[SELECTED]
    df = pd.read_csv(csv_path)
    display(df.head())
    print("Columns:", list(df.columns))
else:
    print("Export/download pre-aligned joint angle CSV files into lunge_sam_out/")


In [ ]:
import re

def recording_identity(path):
    # Prefer explicit subject/trial/camera directories to preserve trial identity.
    relative = path.relative_to(DATA)
    if len(relative.parts) >= 4:
        subject, trial, camera = relative.parts[-4:-1]
        return subject.upper(), trial, camera.lower()
    match = re.fullmatch(r"(s[0-9]+)_(.+)_(cam[0-9]+)_joint_angles[.]csv", path.name, re.I)
    if match:
        subject, trial, camera = match.groups()
        return subject.upper(), trial, camera.lower()
    return None

if df is not None:
    identity = recording_identity(csv_path)
    if identity is None:
        raise ValueError("Use subject/trial/camera directories or subject_trial_camera_joint_angles.csv")
    subject, trial, camera = identity
    out_dir = RESULTS / subject / trial / camera / "legacy_validation"
    if "time_ms_aligned" not in df or not any(
        col.startswith(("vicon_", "r_vicon_")) for col in df
    ):
        raise ValueError("Expected pre-aligned CSV with embedded Vicon. For raw inputs use independent sync JSON and Vicon CSV.")
    metrics = compare_angles(
        csv_path, out_dir, subject=subject, trial=trial, camera=camera,
        allow_legacy_embedded_reference=True, static_window_ms=STATIC_WINDOW_MS,
        plot=False,
    )
    # Consume the engine's canonical output, including explicit right-side references.
    df = pd.read_csv(out_dir / "aligned_angles.csv")
    for joint in ("hip", "knee", "ankle"):
        explicit = f"r_vicon_{joint}_flex"
        if explicit in df:
            df[f"vicon_{joint}_flex"] = df[explicit]
    display(metrics)


In [ ]:
if metrics is not None:
    for plotter, label in ((plot_joint_overlays, "SAM vs Vicon"),
                           (plot_joint_errors, "SAM − Vicon")):
        fig = plotter(df, title=f"{subject} {trial} {camera}: {label}")
        display(fig)
        plt.close(fig)


In [ ]:
if metrics is not None:
    camera_frames = {}
    for path in files:
        identity = recording_identity(path)
        if identity is None or identity[:2] != (subject, trial):
            continue
        frame = pd.read_csv(path)
        if "time_ms_aligned" not in frame:
            continue  # Raw camera clocks cannot be overlaid as aligned recordings.
        camera_name = identity[2]
        if camera_name in camera_frames:
            raise ValueError(f"Duplicate recording for {subject}/{trial}/{camera_name}: {path}")
        for joint in ("hip", "knee", "ankle"):
            explicit = f"r_vicon_{joint}_flex"
            if explicit in frame:
                frame[f"vicon_{joint}_flex"] = frame[explicit]
        camera_frames[camera_name] = frame
    if len(camera_frames) >= 2:
        fig = plot_camera_comparison(camera_frames, joint="knee", method="geo")
        display(fig)
        plt.close(fig)


### Scientific checks
The imported CSVs may already be filtered and interpolated. Do not shift again. Document anatomical signs/definitions, and separately report raw versus independently calibrated error. For real independent validation, use raw SAM `time_ms`, reviewed `sync.json`, and standalone Vicon `time_ms`/angle CSV. Aggregate inference across reps/trials/subjects rather than treating all samples as independent.